# C3 — Barnes-Hut en CUDA (Colab, T4)

Antes de correr: **Entorno de ejecución → Cambiar tipo de entorno → GPU T4**. Correr las celdas en orden. Al terminar: **Archivo → Descargar → .ipynb** y guardarlo en `cuda/bh_cuda.ipynb` del repo (el enunciado pide el notebook ya ejecutado, con salidas).

## 1. La GPU asignada

In [1]:
!nvidia-smi

Wed Nov 25 16:16:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Código
Se clona la rama `impl/cuda` (el repo es público). `make` compila `c_core/` con gcc y `cuda/bh_cuda.cu` con nvcc.

In [2]:
!rm -rf barnes-hut-parallel && git clone -q -b impl/cuda https://github.com/samurestrepo/barnes-hut-parallel.git
%cd barnes-hut-parallel
!git log --oneline -3
!make -C cuda

/content/barnes-hut-parallel
0ee45cb (HEAD -> impl/cuda, origin/impl/cuda) C3: CUDA con árbol aplanado
f0a7898 (origin/impl/c-core) C1: contar interacciones y exponer piezas del árbol para C2
462d5bf C1: Barnes-Hut secuencial en C
make: Entering directory '/content/barnes-hut-parallel/cuda'
gcc -O2 -std=c11 -c ../c_core/bh_io.c -o bh_io.o
gcc -O2 -std=c11 -c ../c_core/bh_arbol.c -o bh_arbol.o
gcc -O2 -std=c11 -c ../c_core/bh_fuerzas.c -o bh_fuerzas.o
nvcc -O2 -arch=sm_75 -lineinfo -o bh_cuda bh_cuda.cu bh_io.o bh_arbol.o bh_fuerzas.o -lm
make: Leaving directory '/content/barnes-hut-parallel/cuda'


## 3. Verificador (umbrales de float32)
Las 5 pruebas del contrato, con 128 hilos por bloque.

In [3]:
!python common/verify.py correr --cmd "./cuda/bh_cuda" --float32 --workers 128

versión: ./cuda/bh_cuda
salidas en: /tmp/verify_dgjcox7g

fuerzas_directa      PASA     1.68e-05 < 1e-03    error relativo máximo de a  [0.5 s]
fuerzas_bh           PASA     1.72e-03 < 1e-02    mediana error relativo de a  [0.4 s]
trayectoria          PASA     3.83e-07 < 1e-03    error máximo de posición  [0.4 s]
energia_reposo       PASA     6.62e-08 < 1e-03    deriva de energía (100 pasos)  [0.6 s]
escrito data/clusters_N2000_s42_v_m.txt (2000 partículas)
energia_movimiento   PASA     6.14e-03 < 2e-02    deriva de energía (100 pasos)  [0.4 s]

RESULTADO: PASA todo


## 4. compute-sanitizer
`memcheck` busca accesos fuera de los arreglos y memoria sin inicializar. `racecheck` solo revisa memoria **compartida** (`__shared__`), que este diseño no usa: la pila de cada hilo es local, por eso se espera 0 avisos.

In [4]:
!compute-sanitizer --tool memcheck ./cuda/bh_cuda --input data/uniform_N2000_s42.txt --steps 2 --workers 128 --out /tmp/cs 2>&1 | tail -5
!compute-sanitizer --tool racecheck ./cuda/bh_cuda --input data/uniform_N2000_s42.txt --steps 1 --workers 128 --out /tmp/cs 2>&1 | tail -3

========= COMPUTE-SANITIZER
N = 2000, pasos = 2, hilos/bloque = 128, orden = morton, theta = 0.5, nodos = 621, total = 0.212 s
========= ERROR SUMMARY: 0 errors
========= COMPUTE-SANITIZER
N = 2000, pasos = 1, hilos/bloque = 128, orden = morton, theta = 0.5, nodos = 621, total = 0.661 s
========= RACECHECK SUMMARY: 0 hazards displayed (0 errors, 0 warnings)


## 5. Desglose por fase y divergencia de warps
N = 200 000, 20 pasos. `ORDEN_HILOS=morton` (vecinos en el espacio en el mismo warp) contra `original` (warps mezclados).

In [5]:
!python common/gen.py --dist uniform --n 200000
!python common/gen.py --dist corner --n 200000
!mkdir -p results/cuda
for dist, dt in [("uniform", "0.001"), ("corner", "2.5e-4")]:
    for orden in ["morton", "original"]:
        pref = f"results/cuda/{dist}_N200000_{orden}_b128"
        !ORDEN_HILOS={orden} ./cuda/bh_cuda --input data/{dist}_N200000_s42.txt --steps 20 --dt {dt} --workers 128 --out {pref}
        !cat {pref}_times.csv
!rm -f results/cuda/*_acc0.txt results/cuda/*_final.txt

escrito data/uniform_N200000_s42.txt (200000 partículas)
escrito data/corner_N200000_s42.txt (200000 partículas)
N = 200000, pasos = 20, hilos/bloque = 128, orden = morton, theta = 0.5, nodos = 69340, total = 3.068 s
fase,segundos
lectura,0.769853
construccion,0.273694
ordenamiento,0.032643
recorrido,1.767654
integracion,0.002230
comunicacion,0.000000
total,3.068287
N = 200000, pasos = 20, hilos/bloque = 128, orden = original, theta = 0.5, nodos = 69340, total = 8.010 s
fase,segundos
lectura,1.294216
construccion,0.208546
ordenamiento,0.025158
recorrido,6.281722
integracion,0.002227
comunicacion,0.000000
total,8.009978
N = 200000, pasos = 20, hilos/bloque = 128, orden = morton, theta = 0.5, nodos = 71830, total = 3.574 s
fase,segundos
lectura,0.732237
construccion,0.284679
ordenamiento,0.025155
recorrido,2.015906
integracion,0.002151
comunicacion,0.000000
total,3.574299
N = 200000, pasos = 20, hilos/bloque = 128, orden = original, theta = 0.5, nodos = 71830, total = 8.934 s
fase,segund

## 6. Tamaño de bloque
64, 128 y 256 hilos por bloque con `common/bench.py` (de Samuel).

In [6]:
!python common/bench.py --impl cuda --etiqueta uniform_N200000_bloques --cmd "./cuda/bh_cuda" --input data/uniform_N200000_s42.txt --trabajadores 64,128,256
!cat results/cuda/uniform_N200000_bloques_bench.csv

procesador guardado en results/lscpu.txt; trabajadores [64, 128, 256]
  k=64 rep=1: total 3.602 s (pared 4.3 s)
  k=128 rep=1: total 3.009 s (pared 3.7 s)
  k=256 rep=1: total 2.960 s (pared 3.7 s)
resumen en results/cuda/uniform_N200000_bloques_bench.csv
trabajadores,repeticion,pared,lectura,construccion,ordenamiento,recorrido,integracion,comunicacion,total
64,1,4.3243,1.108457,0.236449,0.026352,1.881244,0.002171,0.000000,3.602443
128,1,3.7193,0.766066,0.186156,0.023522,1.810543,0.002131,0.000000,3.008931
256,1,3.6796,0.737259,0.184530,0.023184,1.801009,0.002066,0.000000,2.959656


## 7. Comprimir resultados para subirlos al repo
Descargar `resultados_cuda.zip` y descomprimirlo en la raíz del repo.

In [7]:
!cp results/lscpu.txt results/cuda/lscpu_colab.txt; zip -qr resultados_cuda.zip results/cuda && ls -la resultados_cuda.zip
from google.colab import files
files.download("resultados_cuda.zip")

-rw-r--r-- 1 root root 51570551 Nov 25 16:18 resultados_cuda.zip
